# Training the GST Regulatory LLM -- Kaggle

Trains the 131.5M param model on the GST regulatory corpus.

**Data source: Hugging Face, not the Kaggle-attached dataset.** The
Kaggle dataset's remote-URL import silently saved HTML (from a `blob`
link, and separately a fetcher redirect issue) under the `.parquet`
filenames -- so this notebook pulls straight from the HF repo
`Tharun007/gst-rulings-corpus` with `huggingface_hub.hf_hub_download`,
which handles the `resolve` URL and redirects correctly, and verifies
each file's magic bytes after download so a bad fetch fails loudly
instead of surfacing three cells later as an `ArrowInvalid` error.

`/kaggle/working/` persists for the session and is saved when you commit
the notebook, but that isn't the same as Drive's cross-session persistence
-- commit periodically on long runs, don't rely solely on the working
directory surviving an unexpected kernel restart. Kaggle notebooks need
internet access enabled (Settings -> Internet -> On) for the HF download
to work.


In [ ]:
!pip install -q tiktoken huggingface_hub

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

CONTEXT_LEN = 1024
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
MAX_STEPS = 20_000
EVAL_EVERY = 250
EVAL_ITERS = 25
LR = 3e-4
WEIGHT_DECAY = 0.1
PATIENCE = 10

USE_AMP = True
import torch
AMP_DTYPE = torch.float16
USE_ACTIVATION_CHECKPOINTING = True

print("Configuration loaded.")
print(f"Context length       : {CONTEXT_LEN}")
print(f"Micro batch size     : {BATCH_SIZE}")
print(f"Gradient accumulation: {GRAD_ACCUM_STEPS}")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
\n

In [ ]:
import os

# ---- HF dataset location -- EDIT IF THE REPO/FILENAMES CHANGE ----
HF_REPO_ID = "Tharun007/gst-rulings-corpus"
HF_TRAIN_FILE = "data/train-00000-of-00001.parquet"
HF_VAL_FILE = "data/validation-00000-of-00001.parquet"
HF_TOKEN = None  # repo is public -- set a token string only if you make it private

DATA_CACHE_DIR = "/kaggle/working/data"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

os.makedirs(DATA_CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Data cache  -> {DATA_CACHE_DIR}")
print(f"Checkpoints -> {CHECKPOINT_DIR}")


## Download corpus from Hugging Face

Uses `hf_hub_download` (not a raw `requests.get` on a hand-built URL --
that's what produced the HTML-saved-as-.parquet failure earlier). Each
downloaded file's magic bytes are checked before pandas ever touches it,
so a bad fetch throws a clear error here instead of a confusing
`ArrowInvalid` three cells later.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

def _verify_parquet_magic(path):
    """Parquet files start and end with the 4 bytes b'PAR1'. A bad fetch
    (HTML error page, truncated download, etc.) won't have this -- catch
    it here instead of a confusing ArrowInvalid from pandas later."""
    with open(path, "rb") as f:
        head = f.read(4)
        f.seek(-4, os.SEEK_END)
        tail = f.read(4)
    if head != b"PAR1" or tail != b"PAR1":
        preview = open(path, "rb").read(300)
        raise ValueError(
            f"{path} is not a valid parquet file (header={head!r}, footer={tail!r}). "
            f"First 300 bytes:\n{preview}"
        )

def _download_and_verify(filename):
    local_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
        cache_dir=DATA_CACHE_DIR,
    )
    _verify_parquet_magic(local_path)
    return local_path

train_path = _download_and_verify(HF_TRAIN_FILE)
val_path = _download_and_verify(HF_VAL_FILE)
print(f"Train file -> {train_path}")
print(f"Val file   -> {val_path}")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

assert "text" in train_df.columns, f"Expected a 'text' column, got {list(train_df.columns)}"
print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")
print("Sample:", train_df["text"].iloc[0][:200])


## Tokenize + pack

In [ ]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

def tokenize_and_pack(texts, context_len):
    all_ids = []
    for t in texts:
        all_ids.extend(enc.encode(t))
        all_ids.append(EOT)
    arr = np.array(all_ids, dtype=np.uint16)
    n_seq = len(arr) // context_len
    return arr[: n_seq * context_len].reshape(n_seq, context_len)

train_data = tokenize_and_pack(train_df["text"].tolist(), CONTEXT_LEN)
val_data = tokenize_and_pack(val_df["text"].tolist(), CONTEXT_LEN)
print(f"Train sequences: {train_data.shape[0]:,}")
print(f"Val sequences:   {val_data.shape[0]:,}")


## Model architecture

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    context_length: int = 1024
    d_model: int = 768
    n_layers: int = 13
    n_heads: int = 12
    dropout: float = 0.2
    qkv_bias: bool = False

class MultiHeadAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.d_model = cfg.d_model
        self.dropout = cfg.dropout

        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.qkv_bias)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3*d_model)
        q, k, v = qkv.split(self.d_model, dim=2)

        # (B, T, n_heads, head_dim) -> (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v, 
            dropout_p=self.dropout if self.training else 0.0, 
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

def create_block_forward(block):
    def custom_forward(x):
        return block(x)
    return custom_forward

class GPTModel(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying
        self.head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)

        for block in self.blocks:
            if USE_ACTIVATION_CHECKPOINTING and self.training:
                x = checkpoint(create_block_forward(block), x, use_reentrant=False)
            else:
                x = block(x)

        x = self.ln_final(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        return logits, loss

    def num_params(self, exclude_embeddings: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.token_emb.weight.numel()
        return n

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0,
                 top_k: int = None):
        was_training = self.training
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        
        if was_training:
            self.train()
        return idx
\n

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this training setup.")

device = torch.device("cuda:0")
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.2f} GB")

print("Training device:", torch.cuda.get_device_name(device))

cfg = GPTConfig(context_length=CONTEXT_LEN)
model = GPTModel(cfg)

print(f"\nTotal parameters: {model.num_params():,}")
print(f"Parameters excluding tied embedding: {model.num_params(exclude_embeddings=True):,}")

model = model.to(device)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"\nModel device: {next(model.parameters()).device}")
print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
\n

## Training loop

In [ ]:
import numpy as np

def get_batch(data, batch_size, device):
    idx = np.random.randint(0, data.shape[0], size=batch_size)
    seqs = torch.from_numpy(data[idx].astype(np.int64))
    x = seqs[:, :-1].contiguous()
    y = seqs[:, 1:].contiguous()
    return x.to(device), y.to(device)

def configure_optimizer(model, weight_decay, lr):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.dim() >= 2:
            decay_params.append(param)
        else:
            no_decay_params.append(param)
    return torch.optim.AdamW([
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.95))

@torch.no_grad()
def estimate_loss(model, data, batch_size, device, eval_iters=25):
    model.eval()
    losses = []
    for _ in range(eval_iters):
        x, y = get_batch(data, batch_size, device)
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                _, loss = model(x, y)
        else:
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def save_checkpoint(path, model, optimizer, scaler, step, best_val_loss, no_improve_count, cfg):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler else None,
        "step": step,
        "best_val_loss": best_val_loss,
        "no_improve_count": no_improve_count,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "config": vars(cfg)
    }
    torch.save(checkpoint, path)
\n

In [ ]:
import os as _os

last_ckpt_path = _os.path.join(CHECKPOINT_DIR, "last.pt")
best_ckpt_path = _os.path.join(CHECKPOINT_DIR, "best.pt")

optimizer = configure_optimizer(model, WEIGHT_DECAY, LR)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

start_step = 0
best_val_loss = float("inf")
no_improve_count = 0

if _os.path.exists(last_ckpt_path):
    print(f"Found checkpoint:\n{last_ckpt_path}")
    print("Loading checkpoint...")
    ckpt = torch.load(last_ckpt_path, map_location=device)
    
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    
    if "scaler_state_dict" in ckpt and ckpt["scaler_state_dict"] and scaler:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        
    start_step = ckpt.get("step", 0)
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    no_improve_count = ckpt.get("no_improve_count", 0)
    
    if "torch_rng_state" in ckpt:
        torch.set_rng_state(ckpt["torch_rng_state"].cpu())
    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])
        
    print(f"Resumed at step: {start_step}")
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"No improvement count: {no_improve_count}")

    del ckpt
    torch.cuda.empty_cache()
    print(f"CUDA memory allocated after checkpoint cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
else:
    print("No checkpoint found.")
    print("Starting training from scratch.")
\n

In [ ]:
import time

model.train()
t0 = time.time()

optimizer.zero_grad(set_to_none=True)

for step in range(start_step, MAX_STEPS):
    x, y = get_batch(train_data, BATCH_SIZE, device)
    
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            _, loss = model(x, y)
    else:
        _, loss = model(x, y)
        
    loss = loss / GRAD_ACCUM_STEPS
    
    if USE_AMP:
        scaler.scale(loss).backward()
    else:
        loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0 or step == MAX_STEPS - 1:
        if USE_AMP:
            scaler.unscale_(optimizer)
        
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        if USE_AMP:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
            
        optimizer.zero_grad(set_to_none=True)

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        train_loss_est = estimate_loss(model, train_data, BATCH_SIZE, device, EVAL_ITERS)
        val_loss = estimate_loss(model, val_data, BATCH_SIZE, device, EVAL_ITERS)
        elapsed = time.time() - t0
        
        gpu_allocated = torch.cuda.memory_allocated() / 1024**3
        gpu_reserved = torch.cuda.memory_reserved() / 1024**3
        
        print(f"\nstep {step:6d} | train_loss {train_loss_est:.4f} | val_loss {val_loss:.4f} | grad {grad_norm:.2f} | VRAM {gpu_allocated:.2f}/{gpu_reserved:.2f} GB | {elapsed:.0f}s")

        save_checkpoint(last_ckpt_path, model, optimizer, scaler, step, best_val_loss, no_improve_count, cfg)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            save_checkpoint(best_ckpt_path, model, optimizer, scaler, step, best_val_loss, no_improve_count, cfg)
            print(f"  New best val_loss: {best_val_loss:.4f}")
            print(f"  Saved: {best_ckpt_path}")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{PATIENCE})")

        if no_improve_count >= PATIENCE:
            print("\nEarly stopping.")
            print(f"Best validation loss: {best_val_loss:.4f}")
            break
            
        torch.cuda.empty_cache()

print("\nTraining complete (or interrupted).")
print(f"Best validation loss: {best_val_loss:.4f}")
\n